# 62. The Picker Routing Problem

## Tier 3: Firefly Algorithm Implementation

### Key Assumptions
- Firefly brightness is inversely proportional to route distance
- Brighter fireflies (shorter routes) attract dimmer ones
- Attraction force decreases with distance between fireflies
- Random movement prevents premature convergence
- Population diversity maintains exploration capability

### Approach (Step-by-Step)

The Firefly Algorithm mimics natural firefly behavior for optimization:

1. **Initialization**: Create population of random firefly routes
2. **Brightness Calculation**: Evaluate route quality for each firefly
3. **Attraction Movement**: Move dimmer fireflies toward brighter ones
4. **Random Walk**: Add randomization to maintain diversity
5. **Selection**: Keep best solutions and update population
6. **Convergence**: Iterate until stopping criteria met

### What to Look for in Results

- Convergence behavior and iteration progress
- Best solution found vs. theoretical optimum
- Population diversity over time
- Algorithm performance with different parameters
- Comparison with previous tiers (MDP, Divide & Conquer)
- Computational efficiency and solution quality trade-offs

### Concrete Example

We'll implement the 10-item warehouse optimization from the source:
- Warehouse: 15 aisles × 8 cross-aisles
- Items at 10 locations: {(2,3), (5,7), (8,2), (12,6), (3,8), (9,4), (14,1), (6,5), (11,7), (4,2)}
- Population: 30 fireflies, 1000 iterations
- Expected improvement: 26.2% from initial to final solution
- Expected convergence within 847 iterations

In [ ]:
# Import required libraries for Firefly Algorithm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict, Optional
import pandas as pd
import time
import random
from itertools import permutations
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for Firefly Algorithm")

In [ ]:
class Firefly:
    """
    Represents a single firefly in the algorithm
    Each firefly represents a potential picker route solution
    """
    
    def __init__(self, route: List[int], items: Dict[int, Tuple[float, float]], 
                 depot_location: Tuple[float, float]):
        """
        Initialize a firefly with a route
        
        Args:
            route: List of item IDs representing the route
            items: Dictionary mapping item_id to (x, y) coordinates
            depot_location: Starting/ending depot coordinates
        """
        self.route = route.copy()
        self.items = items
        self.depot_location = depot_location
        self.brightness = 0.0
        self.distance = 0.0
        self.calculate_fitness()
    
    def calculate_fitness(self):
        """
        Calculate the fitness (brightness) of this firefly
        Brightness is inversely proportional to route distance
        """
        self.distance = self.calculate_route_distance()
        # Brightness: higher for shorter distances (inverse relationship)
        self.brightness = 1.0 / (1.0 + self.distance)
    
    def calculate_route_distance(self) -> float:
        """
        Calculate total distance for the firefly's route
        
        Returns:
            Total route distance including depot return
        """
        total_distance = 0.0
        
        if not self.route:
            return 0.0
        
        # Distance from depot to first item
        first_coord = self.items[self.route[0]]
        total_distance += np.sqrt((self.depot_location[0] - first_coord[0])**2 + 
                                (self.depot_location[1] - first_coord[1])**2)
        
        # Distance between consecutive items
        for i in range(len(self.route) - 1):
            coord1 = self.items[self.route[i]]
            coord2 = self.items[self.route[i + 1]]
            total_distance += np.sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)
        
        # Distance from last item back to depot
        last_coord = self.items[self.route[-1]]
        total_distance += np.sqrt((last_coord[0] - self.depot_location[0])**2 + 
                                (last_coord[1] - self.depot_location[1])**2)
        
        return total_distance
    
    def copy(self) -> 'Firefly':
        """
        Create a deep copy of this firefly
        
        Returns:
            New firefly with same route
        """
        return Firefly(self.route, self.items, self.depot_location)

class FireflyAlgorithmPickerRouting:
    """
    Firefly Algorithm implementation for the Picker Routing Problem
    
    This nature-inspired metaheuristic uses firefly attraction behavior
    to find high-quality routing solutions.
    """
    
    def __init__(self, items: Dict[int, Tuple[float, float]], 
                 depot_location: Tuple[float, float] = (0, 0),
                 population_size: int = 30, alpha: float = 0.3, 
                 beta0: float = 1.2, gamma: float = 0.02):
        """
        Initialize the Firefly Algorithm
        
        Args:
            items: Dictionary mapping item_id to (x, y) coordinates
            depot_location: Starting/ending depot coordinates
            population_size: Number of fireflies in the population
            alpha: Randomization parameter (0-1)
            beta0: Attraction coefficient at zero distance
            gamma: Light absorption coefficient
        """
        self.items = items
        self.depot_location = depot_location
        self.population_size = population_size
        self.alpha = alpha
        self.beta0 = beta0
        self.gamma = gamma
        
        # Algorithm state
        self.population = []
        self.best_firefly = None
        self.convergence_history = []
        self.diversity_history = []
        self.computation_time = 0.0
        
    def initialize_population(self):
        """
        Initialize the firefly population with random routes
        """
        self.population = []
        item_ids = list(self.items.keys())
        
        for _ in range(self.population_size):
            # Create random permutation of items
            random_route = item_ids.copy()
            random.shuffle(random_route)
            
            firefly = Firefly(random_route, self.items, self.depot_location)
            self.population.append(firefly)
        
        # Find initial best firefly
        self.best_firefly = max(self.population, key=lambda f: f.brightness)
        
        print(f"Initialized population with {self.population_size} fireflies")
        print(f"Initial best distance: {self.best_firefly.distance:.2f}")
    
    def calculate_distance_between_fireflies(self, firefly1: Firefly, firefly2: Firefly) -> float:
        """
        Calculate the distance between two fireflies (solution similarity)
        
        Args:
            firefly1: First firefly
            firefly2: Second firefly
            
        Returns:
            Distance measure between the two solutions
        """
        # Use Hamming distance as a simple measure of route difference
        distance = 0
        min_length = min(len(firefly1.route), len(firefly2.route))
        
        for i in range(min_length):
            if firefly1.route[i] != firefly2.route[i]:
                distance += 1
        
        # Add difference in route lengths
        distance += abs(len(firefly1.route) - len(firefly2.route))
        
        return float(distance)
    
    def move_firefly_toward(self, firefly: Firefly, target_firefly: Firefly, distance: float):
        """
        Move a firefly toward a brighter firefly using attraction
        
        Args:
            firefly: Firefly to move (dimmer)
            target_firefly: Firefly to move toward (brighter)
            distance: Distance between fireflies
        """
        # Calculate attraction coefficient
        beta = self.beta0 * np.exp(-self.gamma * distance ** 2)
        
        # Perform crossover-based movement
        new_route = self.crossover_routes(firefly.route, target_firefly.route, beta)
        
        # Apply random walk for exploration
        if np.random.random() < self.alpha:
            new_route = self.random_walk(new_route)
        
        # Update firefly route
        firefly.route = new_route
        firefly.calculate_fitness()
    
    def crossover_routes(self, route1: List[int], route2: List[int], beta: float) -> List[int]:
        """
        Perform crossover between two routes
        
        Args:
            route1: First parent route
            route2: Second parent route
            beta: Crossover probability
            
        Returns:
            Child route
        """
        if np.random.random() < beta:
            # Order crossover (OX1)
            if len(route1) <= 2:
                return route1.copy()
            
            # Select random crossover points
            start = np.random.randint(0, len(route1) - 1)
            end = np.random.randint(start + 1, len(route1))
            
            # Create child route
            child = [-1] * len(route1)
            
            # Copy segment from route1
            for i in range(start, end):
                child[i] = route1[i]
            
            # Fill remaining positions from route2
            route2_ptr = 0
            for i in range(len(route1)):
                if child[i] == -1:
                    while route2[route2_ptr] in child:
                        route2_ptr += 1
                        if route2_ptr >= len(route2):
                            route2_ptr = 0
                    child[i] = route2[route2_ptr]
                    route2_ptr += 1
            
            return child
        else:
            return route1.copy()
    
    def random_walk(self, route: List[int]) -> List[int]:
        """
        Apply random walk to a route for exploration
        
        Args:
            route: Original route
            
        Returns:
            Modified route after random walk
        """
        if len(route) <= 2:
            return route.copy()
        
        new_route = route.copy()
        
        # Random swap mutation
        if np.random.random() < 0.5:
            i, j = np.random.choice(len(route), 2, replace=False)
            new_route[i], new_route[j] = new_route[j], new_route[i]
        
        # Random inversion mutation
        elif np.random.random() < 0.5:
            i, j = np.random.choice(len(route), 2, replace=False)
            if i > j:
                i, j = j, i
            new_route[i:j+1] = new_route[i:j+1][::-1]
        
        return new_route
    
    def calculate_population_diversity(self) -> float:
        """
        Calculate diversity of the current population
        
        Returns:
            Diversity measure (average distance between fireflies)
        """
        if len(self.population) <= 1:
            return 0.0
        
        total_distance = 0.0
        count = 0
        
        for i in range(len(self.population)):
            for j in range(i + 1, len(self.population)):
                distance = self.calculate_distance_between_fireflies(
                    self.population[i], self.population[j]
                )
                total_distance += distance
                count += 1
        
        return total_distance / count if count > 0 else 0.0
    
    def iterate(self, max_iterations: int = 1000, convergence_threshold: float = 1e-6):
        """
        Run the firefly algorithm for specified iterations
        
        Args:
            max_iterations: Maximum number of iterations
            convergence_threshold: Threshold for early convergence
            
        Returns:
            Dictionary with solution details
        """
        start_time = time.time()
        
        print(f"Starting Firefly Algorithm with {max_iterations} iterations...")
        
        for iteration in range(max_iterations):
            # Sort fireflies by brightness (descending)
            self.population.sort(key=lambda f: f.brightness, reverse=True)
            
            # Move each firefly toward brighter ones
            for i in range(len(self.population)):
                for j in range(i + 1, len(self.population)):
                    # Firefly i is brighter than firefly j
                    distance = self.calculate_distance_between_fireflies(
                        self.population[i], self.population[j]
                    )
                    
                    # Move dimmer firefly toward brighter one
                    self.move_firefly_toward(self.population[j], self.population[i], distance)
            
            # Update best firefly
            current_best = max(self.population, key=lambda f: f.brightness)
            if current_best.brightness > self.best_firefly.brightness:
                self.best_firefly = current_best.copy()
            
            # Record convergence and diversity
            self.convergence_history.append(self.best_firefly.distance)
            self.diversity_history.append(self.calculate_population_diversity())
            
            # Progress reporting
            if iteration % 100 == 0:
                print(f"Iteration {iteration}: Best distance = {self.best_firefly.distance:.2f}, "
                      f"Diversity = {self.diversity_history[-1]:.2f}")
            
            # Check convergence
            if len(self.convergence_history) > 10:
                recent_improvement = abs(self.convergence_history[-1] - self.convergence_history[-10])
                if recent_improvement < convergence_threshold:
                    print(f"Converged at iteration {iteration}")
                    break
        
        self.computation_time = time.time() - start_time
        
        print(f"\nFirefly Algorithm completed in {iteration + 1} iterations")
        print(f"Best distance found: {self.best_firefly.distance:.2f}")
        print(f"Computation time: {self.computation_time:.3f} seconds")
        
        return {
            'best_route': self.best_firefly.route,
            'best_distance': self.best_firefly.distance,
            'iterations': iteration + 1,
            'computation_time': self.computation_time,
            'convergence_history': self.convergence_history,
            'diversity_history': self.diversity_history
        }

print("FireflyAlgorithmPickerRouting class defined successfully")

In [ ]:
# Create the concrete example from the problem description
# 10-item warehouse optimization

# Define items with their coordinates (from problem description)
items = {
    1: (2, 3),   # Item 1
    2: (5, 7),   # Item 2
    3: (8, 2),   # Item 3
    4: (12, 6),  # Item 4
    5: (3, 8),   # Item 5
    6: (9, 4),   # Item 6
    7: (14, 1),  # Item 7
    8: (6, 5),   # Item 8
    9: (11, 7),  # Item 9
    10: (4, 2),  # Item 10
}

depot_location = (0, 0)

print("=== Picker Routing Problem: Firefly Algorithm ===")
print(f"\nDepot location: {depot_location}")
print(f"\nItems to collect:")
for item_id, coord in items.items():
    print(f"  Item {item_id}: {coord}")

# Create and run Firefly Algorithm
firefly_solver = FireflyAlgorithmPickerRouting(
    items=items,
    depot_location=depot_location,
    population_size=30,
    alpha=0.3,
    beta0=1.2,
    gamma=0.02
)

# Initialize population
firefly_solver.initialize_population()

# Run the algorithm
solution = firefly_solver.iterate(max_iterations=1000)

In [ ]:
# Analyze the solution
print("\n=== Solution Analysis ===")
print(f"Best route: {solution['best_route']}")
print(f"Best distance: {solution['best_distance']:.2f}")
print(f"Iterations to convergence: {solution['iterations']}")
print(f"Computation time: {solution['computation_time']:.3f} seconds")

# Calculate improvement metrics
initial_distance = firefly_solver.convergence_history[0]
final_distance = solution['best_distance']
improvement = (initial_distance - final_distance) / initial_distance * 100

print(f"\n=== Performance Metrics ===")
print(f"Initial best distance: {initial_distance:.2f}")
print(f"Final best distance: {final_distance:.2f}")
print(f"Total improvement: {improvement:.1f}%")

# Verify against expected solution from problem description
expected_improvement = 26.2  # Expected 26.2% improvement
expected_iterations = 847    # Expected convergence within 847 iterations

print(f"\n=== Verification Against Expected Results ===")
print(f"Expected improvement: {expected_improvement}%")
print(f"Actual improvement: {improvement:.1f}%")
print(f"Expected iterations: ≤{expected_iterations}")
print(f"Actual iterations: {solution['iterations']}")

improvement_match = abs(improvement - expected_improvement) <= 2.0  # Within 2% tolerance
iteration_match = solution['iterations'] <= expected_iterations

print(f"\nImprovement matches expected: {improvement_match}")
print(f"Convergence within expected: {iteration_match}")

# Display route details with coordinates
print(f"\n=== Detailed Route Analysis ===")
print("Step | Item | Coordinates | Cumulative Distance")
print("-" * 50)

cumulative_distance = 0.0
current_pos = depot_location

for step, item_id in enumerate(solution['best_route']):
    item_coord = items[item_id]
    step_distance = np.sqrt((current_pos[0] - item_coord[0])**2 + 
                           (current_pos[1] - item_coord[1])**2)
    cumulative_distance += step_distance
    
    print(f"{step+1:4d} | {item_id:4d} | {item_coord} | {cumulative_distance:8.2f}")
    current_pos = item_coord

# Add return to depot
return_distance = np.sqrt((current_pos[0] - depot_location[0])**2 + 
                          (current_pos[1] - depot_location[1])**2)
cumulative_distance += return_distance
print(f"{len(solution['best_route'])+1:4d} | {'Depot':4s} | {depot_location} | {cumulative_distance:8.2f}")

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Picker Routing Problem: Firefly Algorithm Analysis', fontsize=16, fontweight='bold')

# 1. Convergence Plot
ax1 = axes[0, 0]
iterations = range(len(solution['convergence_history']))
ax1.plot(iterations, solution['convergence_history'], 'b-', linewidth=2, alpha=0.7)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Best Route Distance')
ax1.set_title('Convergence Behavior')
ax1.grid(True, alpha=0.3)

# Add improvement annotation
initial_dist = solution['convergence_history'][0]
final_dist = solution['convergence_history'][-1]
ax1.annotate(f'Improvement: {improvement:.1f}%', 
             xy=(len(iterations)//2, (initial_dist + final_dist)/2),
             xytext=(len(iterations)//2, (initial_dist + final_dist)/2 + 5),
             ha='center', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# 2. Route Visualization
ax2 = axes[0, 1]
route = solution['best_route']

# Plot depot
ax2.plot(depot_location[0], depot_location[1], 'ks', markersize=12, label='Depot')

# Plot items
for item_id, coord in items.items():
    ax2.plot(coord[0], coord[1], 'ro', markersize=8)
    ax2.annotate(f'{item_id}', (coord[0], coord[1]), xytext=(coord[0]+0.2, coord[1]+0.2),
                fontsize=9, fontweight='bold')

# Plot route path
route_coords = [depot_location]
for item_id in route:
    route_coords.append(items[item_id])
route_coords.append(depot_location)

for i in range(len(route_coords) - 1):
    x_coords = [route_coords[i][0], route_coords[i+1][0]]
    y_coords = [route_coords[i][1], route_coords[i+1][1]]
    ax2.plot(x_coords, y_coords, 'b-', linewidth=2, alpha=0.7)
    
    # Add arrow to show direction
    if i < len(route_coords) - 2:
        mid_x = (x_coords[0] + x_coords[1]) / 2
        mid_y = (y_coords[0] + y_coords[1]) / 2
        dx = x_coords[1] - x_coords[0]
        dy = y_coords[1] - y_coords[0]
        ax2.arrow(mid_x - dx*0.1, mid_y - dy*0.1, dx*0.2, dy*0.2, 
                 head_width=0.3, head_length=0.2, fc='blue', ec='blue', alpha=0.5)

ax2.set_xlabel('X Coordinate')
ax2.set_ylabel('Y Coordinate')
ax2.set_title(f'Optimal Route (Distance: {solution["best_distance"]:.2f})')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

# 3. Population Diversity Over Time
ax3 = axes[1, 0]
diversity_iterations = range(len(solution['diversity_history']))
ax3.plot(diversity_iterations, solution['diversity_history'], 'g-', linewidth=2, alpha=0.7)
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Population Diversity')
ax3.set_title('Population Diversity Evolution')
ax3.grid(True, alpha=0.3)

# Add diversity trend analysis
if len(solution['diversity_history']) > 10:
    initial_diversity = solution['diversity_history'][0]
    final_diversity = solution['diversity_history'][-1]
    diversity_reduction = (initial_diversity - final_diversity) / initial_diversity * 100
    ax3.annotate(f'Diversity reduction: {diversity_reduction:.1f}%', 
                 xy=(len(diversity_iterations)//2, (initial_diversity + final_diversity)/2),
                 xytext=(len(diversity_iterations)//2, (initial_diversity + final_diversity)/2 + 1),
                 ha='center', fontsize=10, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))

# 4. Algorithm Performance Comparison
ax4 = axes[1, 1]

# Compare with expected performance and other methods
methods = ['Initial Random', 'Firefly Algorithm', 'Divide & Conquer', 'MDP (Optimal)']
distances = [initial_distance, solution['best_distance'], 26.1, 24.8]  # Estimated values
times = [0.001, solution['computation_time'], 0.23, 4.7]  # From problem descriptions

ax4_twin = ax4.twinx()

# Plot distances
bars1 = ax4.bar([i-0.2 for i in range(len(methods))], distances, 0.4, 
                 label='Route Distance', alpha=0.7, color='skyblue')
ax4.set_ylabel('Route Distance', color='skyblue')
ax4.tick_params(axis='y', labelcolor='skyblue')

# Plot computation times
bars2 = ax4_twin.bar([i+0.2 for i in range(len(methods))], times, 0.4,
                      label='Computation Time (s)', alpha=0.7, color='lightcoral')
ax4_twin.set_ylabel('Computation Time (seconds)', color='lightcoral')
ax4_twin.tick_params(axis='y', labelcolor='lightcoral')

ax4.set_xlabel('Method')
ax4.set_title('Algorithm Performance Comparison')
ax4.set_xticks(range(len(methods)))
ax4.set_xticklabels(methods, rotation=45, ha='right')
ax4.grid(True, alpha=0.3)

# Add value labels
for bar, value in zip(bars1, distances):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{value:.1f}', ha='center', va='bottom', fontsize=8)

for bar, value in zip(bars2, times):
    ax4_twin.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                  f'{value:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print("\n=== Visualization Summary ===")
print("1. Convergence shows steady improvement over iterations")
print("2. Optimal route efficiently connects all warehouse locations")
print("3. Population diversity decreases as algorithm converges")
print("4. Firefly Algorithm balances solution quality and computation time")

In [ ]:
# Parameter Sensitivity Analysis
print("\n=== Parameter Sensitivity Analysis ===")

def test_parameter_combinations():
    """
    Test different parameter combinations for the Firefly Algorithm
    """
    # Define parameter ranges to test
    alphas = [0.1, 0.3, 0.5]
    beta0s = [0.8, 1.2, 1.6]
    gammas = [0.01, 0.02, 0.04]
    
    results = []
    
    for alpha in alphas:
        for beta0 in beta0s:
            for gamma in gammas:
                print(f"\nTesting: α={alpha}, β₀={beta0}, γ={gamma}")
                
                # Create solver with test parameters
                test_solver = FireflyAlgorithmPickerRouting(
                    items=items,
                    depot_location=depot_location,
                    population_size=20,  # Smaller for faster testing
                    alpha=alpha,
                    beta0=beta0,
                    gamma=gamma
                )
                
                test_solver.initialize_population()
                test_solution = test_solver.iterate(max_iterations=500)  # Fewer iterations for testing
                
                results.append({
                    'alpha': alpha,
                    'beta0': beta0,
                    'gamma': gamma,
                    'best_distance': test_solution['best_distance'],
                    'iterations': test_solution['iterations'],
                    'time': test_solution['computation_time']
                })
                
                print(f"  Distance: {test_solution['best_distance']:.2f}")
                print(f"  Iterations: {test_solution['iterations']}")
    
    return results

# Run parameter sensitivity test
sensitivity_results = test_parameter_combinations()

# Create parameter sensitivity visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Firefly Algorithm: Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')

# Convert results to DataFrame for easier analysis
df = pd.DataFrame(sensitivity_results)

# 1. Alpha parameter impact
ax1 = axes[0, 0]
alpha_groups = df.groupby('alpha')['best_distance'].mean()
ax1.bar(alpha_groups.index, alpha_groups.values, alpha=0.7, color='skyblue')
ax1.set_xlabel('Alpha (α) - Randomization')
ax1.set_ylabel('Average Best Distance')
ax1.set_title('Impact of Randomization Parameter')
ax1.grid(True, alpha=0.3)

# 2. Beta0 parameter impact
ax2 = axes[0, 1]
beta0_groups = df.groupby('beta0')['best_distance'].mean()
ax2.bar(beta0_groups.index, beta0_groups.values, alpha=0.7, color='lightcoral')
ax2.set_xlabel('Beta₀ - Initial Attraction')
ax2.set_ylabel('Average Best Distance')
ax2.set_title('Impact of Attraction Coefficient')
ax2.grid(True, alpha=0.3)

# 3. Gamma parameter impact
ax3 = axes[1, 0]
gamma_groups = df.groupby('gamma')['best_distance'].mean()
ax3.bar(gamma_groups.index, gamma_groups.values, alpha=0.7, color='lightgreen')
ax3.set_xlabel('Gamma (γ) - Light Absorption')
ax3.set_ylabel('Average Best Distance')
ax3.set_title('Impact of Light Absorption Coefficient')
ax3.grid(True, alpha=0.3)

# 4. Convergence speed vs solution quality
ax4 = axes[1, 1]
scatter = ax4.scatter(df['iterations'], df['best_distance'], 
                    c=df['time'], s=100, alpha=0.6, cmap='viridis')
ax4.set_xlabel('Iterations to Convergence')
ax4.set_ylabel('Best Distance Found')
ax4.set_title('Convergence Speed vs Solution Quality')
ax4.grid(True, alpha=0.3)

# Add colorbar for computation time
cbar = plt.colorbar(scatter, ax=ax4)
cbar.set_label('Computation Time (s)')

plt.tight_layout()
plt.show()

# Find best parameter combination
best_result = min(sensitivity_results, key=lambda x: x['best_distance'])
print(f"\n=== Best Parameter Combination ===")
print(f"Alpha: {best_result['alpha']}")
print(f"Beta₀: {best_result['beta0']}")
print(f"Gamma: {best_result['gamma']}")
print(f"Best distance: {best_result['best_distance']:.2f}")
print(f"Iterations: {best_result['iterations']}")
print(f"Time: {best_result['time']:.3f}s")

# Display sensitivity results summary
print(f"\n=== Parameter Sensitivity Summary ===")
print("Parameter | Best Distance | Avg Iterations | Avg Time (s)")
print("-" * 55)

for param in ['alpha', 'beta0', 'gamma']:
    param_groups = df.groupby(param)
    for value, group in param_groups:
        avg_distance = group['best_distance'].mean()
        avg_iterations = group['iterations'].mean()
        avg_time = group['time'].mean()
        print(f"{param:9s}={value:5.1f} | {avg_distance:13.2f} | {avg_iterations:14.1f} | {avg_time:11.3f}")

### Why This Tier Exists vs Previous Tiers

**Tier 3 (Firefly Algorithm)** provides advanced metaheuristic capabilities beyond exact and heuristic methods:

**Advantages over Tier 1 (MDP):**
- **Scalability**: Handles large problems (20+ items) vs MDP limit (~10 items)
- **Population-Based Search**: Explores multiple solutions simultaneously
- **Global Optimization**: Less likely to get stuck in local optima
- **Adaptive Behavior**: Dynamic attraction and exploration mechanisms
- **Parallel Processing**: Natural parallelization of firefly evaluations

**Advantages over Tier 2 (Divide & Conquer):**
- **Solution Quality**: Often finds better solutions than partition-based methods
- **No Partition Dependency**: Doesn't rely on warehouse structure
- **Swarm Intelligence**: Collective behavior emerges from simple rules
- **Flexibility**: Adapts to different problem structures automatically
- **Parameter Tuning**: Can be optimized for specific problem types

**Disadvantages vs Previous Tiers:**
- **Parameter Sensitivity**: Performance depends on proper parameter tuning
- **Computational Cost**: More expensive than simple heuristics
- **Convergence Uncertainty**: May not guarantee convergence to optimal
- **Complexity**: More complex to implement and understand

**When to Use This Tier:**
- Medium to large warehouses (10-50 items)
- When solution quality is more important than speed
- When problem structure doesn't favor partitioning
- For research and algorithm development
- When multiple high-quality solutions are needed

**Performance Verification:**
- Expected improvement: 26.2%, Actual: {improvement:.1f}%
- Expected iterations: ≤847, Actual: {solution['iterations']}
- Solution quality: Within 3.2% of optimal (as stated in problem)
- Computational efficiency: Suitable for medium-scale problems

**Algorithm Characteristics:**
- **Swarm Intelligence**: Fireflies communicate through light intensity
- **Attraction Mechanism**: Brighter solutions attract dimmer ones
- **Balance Exploration-Exploitation**: Random walks maintain diversity
- **Convergence Behavior**: Gradual improvement with diversity reduction

**Comparison Summary:**
- **Tier 1**: Optimal but limited to small problems
- **Tier 2**: Scalable but partition-dependent
- **Tier 3**: High-quality solutions for medium problems with swarm intelligence

The Firefly Algorithm demonstrates how nature-inspired metaheuristics can provide effective solutions for complex routing problems, balancing exploration and exploitation through swarm intelligence mechanisms.